# Notebook 4: Innovation 1 — XLM-RoBERTa

## What is this innovation?

The original paper uses **mBERT** as the multilingual model. mBERT was trained in 2019.

We replace it with **XLM-RoBERTa (XLM-R)**, released in 2020 by Conneau et al.
XLM-R is strictly superior:
- Trained on **2.5 TB** of data vs mBERT's ~1 TB
- Uses **SentencePiece** tokenization (better coverage of rare languages)
- Uses **RoBERTa** training objective (no NSP, more masking, longer training)
- Covers **100 languages** with better balance

## Why is this a valid thesis contribution?

The paper only tested with mBERT. Key questions we answer:
1. Does LAPT still help when starting from a stronger base model (XLM-R)?
2. Does the benefit of LAPT correlate with language type (Type 0/1/2) for XLM-R too?
3. Does XLM-R make VA less necessary (since its tokenizer has better language coverage)?

## Expected hypothesis

- XLM-R baseline > mBERT baseline for all languages
- LAPT still helps XLM-R, especially for low-resource languages
- The gap between XLM-R baseline and XLM-R+LAPT may be smaller than for mBERT
  (because XLM-R already has better coverage)

In [ ]:
# Environment setup
import os, sys, json
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR  = '/content/parsing-mbert'
WORKSPACE = '/content/drive/MyDrive/thesis_mbert'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone https://github.com/Sansgithub22/mtechthesis2Parsing-mbert.git {REPO_DIR}')

sys.path.insert(0, os.path.join(REPO_DIR, 'thesis', 'src'))

# Colab already has PyTorch — only install the missing packages
!pip install -q transformers==4.38.0 tokenizers==0.15.2 datasets==2.18.0 peft==0.10.0 accelerate==0.27.0 conllu==4.5.3 sentencepiece==0.1.99 tqdm seaborn

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch: {torch.__version__} | Device: {DEVICE}')

In [ ]:
# Data paths (same as before)
UD_BASE = os.path.join(WORKSPACE, 'ud-treebanks-v2.5')
TREEBANKS = {
    'ga':   {'train': f'{UD_BASE}/UD_Irish-IDT/ga_idt-ud-train.conllu',
             'dev':   f'{UD_BASE}/UD_Irish-IDT/ga_idt-ud-dev.conllu',
             'test':  f'{UD_BASE}/UD_Irish-IDT/ga_idt-ud-test.conllu'},
    'mt':   {'train': f'{UD_BASE}/UD_Maltese-MUDT/mt_mudt-ud-train.conllu',
             'dev':   f'{UD_BASE}/UD_Maltese-MUDT/mt_mudt-ud-dev.conllu',
             'test':  f'{UD_BASE}/UD_Maltese-MUDT/mt_mudt-ud-test.conllu'},
    'vi':   {'train': f'{UD_BASE}/UD_Vietnamese-VTB/vi_vtb-ud-train.conllu',
             'dev':   f'{UD_BASE}/UD_Vietnamese-VTB/vi_vtb-ud-dev.conllu',
             'test':  f'{UD_BASE}/UD_Vietnamese-VTB/vi_vtb-ud-test.conllu'},
    'sing': {'train': f'{REPO_DIR}/data/sing/train.conllu',
             'dev':   f'{REPO_DIR}/data/sing/dev.conllu',
             'test':  f'{REPO_DIR}/data/sing/test.conllu'},
}
UNLABELED = {
    'ga':   f'{REPO_DIR}/data/unlabeled/ga',
    'mt':   f'{REPO_DIR}/data/unlabeled/mt',
    'vi':   f'{REPO_DIR}/data/unlabeled/vi',
    'sing': f'{REPO_DIR}/data/unlabeled/sg',
}
LAPT_EPOCHS = {'ga': 5, 'mt': 20, 'vi': 5, 'sing': 5}
print('Configured.')

In [ ]:
# Shared experiment runner
from transformers import AutoTokenizer
from data.conllu_reader import read_conllu, get_relation_vocab
from models.encoder import BERTEncoder
from models.biaffine_parser import BiaffineParser
from training.trainer import ParserTrainer

def run_experiment(lang, model_path, freeze_bert, experiment_name,
                   max_epochs=200, patience=20, batch_size=16,
                   bert_lr=5e-5, parser_lr=1e-3):
    print(f'\n{"="*60}\n{experiment_name} | {lang.upper()}\n{"="*60}\n')
    paths = TREEBANKS[lang]
    train_sents = read_conllu(paths['train'])
    dev_sents   = read_conllu(paths['dev'])
    test_sents  = read_conllu(paths['test'])
    rel_vocab   = get_relation_vocab(train_sents)
    n_rels      = max(rel_vocab.values()) + 1

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    encoder   = BERTEncoder(model_name=model_path, freeze=freeze_bert, dropout=0.1)
    model     = BiaffineParser(encoder, n_rels, arc_dim=500, rel_dim=100,
                               bilstm_layers=3, bilstm_hidden=400,
                               mlp_dropout=0.33, use_bilstm=freeze_bert)

    save_dir = os.path.join(WORKSPACE, 'results', experiment_name, lang)
    os.makedirs(save_dir, exist_ok=True)

    trainer = ParserTrainer(
        model, train_sents, dev_sents, tokenizer, rel_vocab, save_dir,
        batch_size=batch_size, max_epochs=max_epochs, patience=patience,
        bert_lr=bert_lr, parser_lr=parser_lr, device=DEVICE)

    best_dev_las = trainer.train()
    test_las, test_uas = trainer.evaluate_test(test_sents, tokenizer)

    result = {'lang': lang, 'experiment': experiment_name,
              'best_dev_las': round(best_dev_las, 2),
              'test_las': round(test_las, 2), 'test_uas': round(test_uas, 2)}
    with open(os.path.join(save_dir, 'result.json'), 'w') as f:
        json.dump(result, f, indent=2)
    return result

print('Runner loaded.')

In [ ]:
# ============================================================
# EXPERIMENT A: XLM-R Baseline (no adaptation)
# ============================================================
# Same as mBERT baseline, but use XLM-R instead.
# This is the simplest innovation: just swap the backbone.

LANG = 'mt'   # Change to run different languages
XLM_R_MODEL = 'xlm-roberta-base'

# XLM-R Frozen
xlmr_frozen = run_experiment(
    lang=LANG,
    model_path=XLM_R_MODEL,
    freeze_bert=True,
    experiment_name='xlmr_frozen',
)
print(f'XLM-R frozen [{LANG}]: LAS = {xlmr_frozen["test_las"]}')

In [ ]:
# XLM-R + FT
xlmr_ft = run_experiment(
    lang=LANG,
    model_path=XLM_R_MODEL,
    freeze_bert=False,
    experiment_name='xlmr_ft',
)
print(f'XLM-R+FT [{LANG}]: LAS = {xlmr_ft["test_las"]}')

In [ ]:
# ============================================================
# EXPERIMENT B: XLM-R + LAPT
# ============================================================
# Run LAPT on XLM-R, then train the parser.
# Key question: does LAPT still help XLM-R as much as it helped mBERT?

from training.lapt import run_lapt

xlmr_lapt_dir = os.path.join(WORKSPACE, 'lapt_models', f'xlmr_lapt_{LANG}')

if os.path.exists(os.path.join(xlmr_lapt_dir, 'config.json')):
    print(f'XLM-R LAPT model already exists for {LANG}')
else:
    print(f'Running LAPT on XLM-R for {LANG.upper()}...')
    run_lapt(
        model_name_or_path=XLM_R_MODEL,
        train_text_path=os.path.join(UNLABELED[LANG], 'train.txt'),
        eval_text_path=os.path.join(UNLABELED[LANG], 'valid.txt'),
        output_dir=xlmr_lapt_dir,
        num_epochs=LAPT_EPOCHS[LANG],
        batch_size=16,
        learning_rate=2e-5,
        fp16=True,
    )

# Train parser on XLM-R+LAPT
xlmr_lapt_ft = run_experiment(
    lang=LANG,
    model_path=xlmr_lapt_dir,
    freeze_bert=False,
    experiment_name='xlmr_lapt_ft',
)
print(f'XLM-R+LAPT+FT [{LANG}]: LAS = {xlmr_lapt_ft["test_las"]}')

In [ ]:
# ============================================================
# Comparison Table: mBERT vs XLM-R
# ============================================================

LANG_NAMES = {'ga': 'Irish', 'mt': 'Maltese', 'vi': 'Vietnamese', 'sing': 'Singlish'}
LANG_TYPE  = {'ga': 'Type 1', 'mt': 'Type 2', 'vi': 'Type 0', 'sing': 'Type 0'}

PAPER_MBERT = {'ga': 68.19, 'mt': 67.06, 'vi': 62.96, 'sing': 74.01}
PAPER_MBERT_FT = {'ga': 72.67, 'mt': 76.74, 'vi': 66.13, 'sing': 78.24}
PAPER_LAPT_FT  = {'ga': 75.45, 'mt': 82.77, 'vi': 67.50, 'sing': 79.30}

print(f'\nInnovation 1 Results: XLM-R vs mBERT')
print(f'{"="*85}')
print(f'{"Method":<25} {"GA (T1)":>10} {"MT (T2)":>10} {"SING (T0)":>10} {"VI (T0)":>10}')
print(f'{"-"*85}')

# Paper baselines
print(f'{"Paper: MBERT frozen":<25}',
      *[f'{PAPER_MBERT.get(l, "N/A"):>10}' for l in ["ga","mt","sing","vi"]])
print(f'{"Paper: MBERT+FT":<25}',
      *[f'{PAPER_MBERT_FT.get(l, "N/A"):>10}' for l in ["ga","mt","sing","vi"]])
print(f'{"Paper: LAPT+FT":<25}',
      *[f'{PAPER_LAPT_FT.get(l, "N/A"):>10}' for l in ["ga","mt","sing","vi"]])
print(f'{"-"*85}')

# Our XLM-R results
for exp_name, label in [('xlmr_frozen','XLM-R frozen'), ('xlmr_ft','XLM-R+FT'), ('xlmr_lapt_ft','XLM-R+LAPT+FT')]:
    row = []
    for lang in ['ga','mt','sing','vi']:
        path = os.path.join(WORKSPACE, 'results', exp_name, lang, 'result.json')
        if os.path.exists(path):
            with open(path) as f:
                row.append(f'{json.load(f)["test_las"]:>10}')
        else:
            row.append(f'{"N/A":>10}')
    print(f'{label:<25}', *row)

print(f'{"="*85}')
print('\nKey research questions to address in your thesis:')
print('1. Is XLM-R baseline consistently better than mBERT baseline?')
print('2. Does LAPT help XLM-R as much as it helped mBERT?')
print('3. Is the improvement pattern still correlated with language type (T0/T1/T2)?')